# 02 - Feature engineering

Builds datetime, road-type, visibility, weather-index, geographic and text features, the composite `Speed_Violation_Risk_Score`, the binary target (Low = Severity 1-2, High = Severity 3-4), and the one-hot / frequency / log encodings.

**Run after 01.** Input: `outputs/Processed_Data/cleaned_dataset.csv`. Output: `outputs/Processed_Data/processed_full_dataset.csv`.

*Source: `CODE/02_feature_engineering.py`*

## 0. Environment

In [1]:
import sys, importlib
print("Python", sys.version.split()[0])
for pkg in ['pandas', 'numpy']:
    try:
        mod = importlib.import_module(pkg)
        print(f"{pkg:<12}{getattr(mod, '__version__', '?')}")
    except ImportError:
        print(f"{pkg:<12}NOT INSTALLED")

Python 3.13.18
pandas      2.2.3
numpy       2.1.3


## 1. Imports and configuration

In [2]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)  # display only (e.g. seaborn palette notice)
warnings.filterwarnings("ignore", message=".*'str' dtypes are included.*")  # display only (pandas 3 select_dtypes notice)

import os
import numpy as np
import pandas as pd

# ---- values below are copied verbatim from config.py (inlined so this notebook runs standalone) ----
RANDOM_SEED = 42
RAW_DATA_PATH = "US A 10000.csv"
PROCESSED_DATA_PATH = "processed_full_dataset.csv"
TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
TEST_SIZE = 0.15
CV_FOLDS = 5
MIN_ACCEPTABLE_ACCURACY = 80.0
MAX_ACCEPTABLE_ACCURACY = 97.0
MAX_TRAIN_TEST_GAP = 5.0
TARGET_COL_ORIGINAL = "Severity"
TARGET_COL_BINARY = "Severity_Binary"
OUTPUT_DIR = "outputs"
FIGURES_DIR = "outputs/Figures"
TABLES_DIR = "outputs/Tables"
MODELS_DIR = "outputs/Models"

np.random.seed(RANDOM_SEED)
ROAD_INFRA_COLS = [
    "Amenity", "Bump", "Crossing", "Give_Way", "Junction", "No_Exit",
    "Railway", "Roundabout", "Station", "Stop", "Traffic_Calming",
    "Traffic_Signal", "Turning_Loop",
]
SPEED_REDUCING_COLS = [
    "Traffic_Calming", "Traffic_Signal", "Stop", "Crossing",
    "Give_Way", "Roundabout",
]
HIGH_FREQ_CARD_COLS = ["Street", "City", "County", "Zipcode", "Airport_Code", "Country"]
WIND_DIR_COL = "Wind_Direction"
WEATHER_COL = "Weather_Condition"
TWILIGHT_COLS = ["Sunrise_Sunset", "Civil_Twilight", "Nautical_Twilight", "Astronomical_Twilight"]
LOG_TRANSFORM_COLS = [
    "Distance(mi)", "Precipitation(in)", "Description_Length",
    "Description_WordCount", "Duration_Minutes",
]

## 2. Functions (verbatim from the original script)

### `add_datetime_features()`

In [3]:
def add_datetime_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["Start_Time"] = pd.to_datetime(df["Start_Time"], errors="coerce")

    df["Hour"] = df["Start_Time"].dt.hour
    df["DayOfWeek"] = df["Start_Time"].dt.dayofweek
    df["Month"] = df["Start_Time"].dt.month
    df["Year"] = df["Start_Time"].dt.year

    df["Is_Weekend"] = df["DayOfWeek"].isin([5, 6]).astype(int)
    df["Is_Night"] = df["Hour"].apply(lambda h: 1 if (pd.isna(h) or h >= 20 or h < 6) else 0)
    df["Is_RushHour"] = df["Hour"].apply(
        lambda h: 1 if (not pd.isna(h) and ((7 <= h <= 9) or (16 <= h <= 18))) else 0
    )
    return df

### `add_road_type_features()`

In [4]:
def add_road_type_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    street = df["Street"].astype(str).str.upper() if "Street" in df.columns else pd.Series("", index=df.index)

    df["Is_Interstate"] = street.str.contains(r"\bI-?\d+", regex=True, na=False).astype(int)
    df["Is_USHighway"] = street.str.contains(r"\bUS-?\d+|HWY", regex=True, na=False).astype(int)
    df["Is_StateRoute"] = street.str.contains(r"\bSR-?\d+|STATE ROUTE", regex=True, na=False).astype(int)
    df["Is_HighSpeedRoad"] = ((df["Is_Interstate"] == 1) | (df["Is_USHighway"] == 1)).astype(int)

    present_infra = [c for c in ROAD_INFRA_COLS if c in df.columns]
    df["Traffic_Control_Count"] = df[present_infra].astype(int).sum(axis=1) if present_infra else 0

    present_reducing = [c for c in SPEED_REDUCING_COLS if c in df.columns]
    df["Has_Speed_Reducing_Feature"] = (
        (df[present_reducing].astype(int).sum(axis=1) > 0).astype(int) if present_reducing else 0
    )
    df["No_Traffic_Control"] = (df["Traffic_Control_Count"] == 0).astype(int)
    return df

### `add_visibility_features()`

In [5]:
def add_visibility_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "Visibility(mi)" in df.columns:
        df["Low_Visibility"] = (df["Visibility(mi)"] < 2.0).astype(int)
    else:
        df["Low_Visibility"] = 0
    return df

### `add_speed_violation_risk_score()`

Composite proxy for speed-violation risk. The raw dataset has no

In [6]:
def add_speed_violation_risk_score(df: pd.DataFrame) -> pd.DataFrame:
    """
    Composite proxy for speed-violation risk. The raw dataset has no
    explicit speed-limit/violation field, so this score combines four
    weighted risk drivers:
        - high-speed road class (Interstate/US Highway)
        - absence of speed-reducing infrastructure
        - night-time driving
        - low-visibility conditions
    Weighted sum, then normalized to a 0-1 scale.
    """
    df = df.copy()
    weights = {
        "Is_HighSpeedRoad": 0.35,
        "No_Traffic_Control": 0.25,
        "Is_Night": 0.25,
        "Low_Visibility": 0.15,
    }
    score = sum(df[col].astype(float) * w for col, w in weights.items())
    df["Speed_Violation_Risk_Score"] = score / sum(weights.values())
    df["High_Speed_Violation_Risk"] = (
        df["Speed_Violation_Risk_Score"] >= df["Speed_Violation_Risk_Score"].quantile(0.75)
    ).astype(int)
    return df

### `add_weather_severity_index()`

In [7]:
def add_weather_severity_index(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    parts = []
    if "Low_Visibility" in df.columns:
        parts.append(df["Low_Visibility"] * 0.4)
    if "Precipitation(in)" in df.columns:
        precip_norm = (df["Precipitation(in)"] - df["Precipitation(in)"].min()) / (
            df["Precipitation(in)"].max() - df["Precipitation(in)"].min() + 1e-9
        )
        parts.append(precip_norm * 0.35)
    if "Wind_Speed(mph)" in df.columns:
        wind_norm = (df["Wind_Speed(mph)"] - df["Wind_Speed(mph)"].min()) / (
            df["Wind_Speed(mph)"].max() - df["Wind_Speed(mph)"].min() + 1e-9
        )
        parts.append(wind_norm * 0.25)
    df["Weather_Severity_Index"] = sum(parts) if parts else 0.0
    return df

### `add_geo_and_text_features()`

In [8]:
def add_geo_and_text_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if {"Start_Lat", "End_Lat"}.issubset(df.columns):
        df["Lat_Diff"] = (df["End_Lat"] - df["Start_Lat"]).abs()
    if {"Start_Lng", "End_Lng"}.issubset(df.columns):
        df["Lng_Diff"] = (df["End_Lng"] - df["Start_Lng"]).abs()

    if "Description" in df.columns:
        desc = df["Description"].astype(str)
        df["Description_Length"] = desc.str.len()
        df["Description_WordCount"] = desc.str.split().str.len()
    return df

### `add_target_binary()`

Low: Severity 1-2 -> 0, High: Severity 3-4 -> 1 (classes 1 & 4 are extremely rare).

In [9]:
def add_target_binary(df: pd.DataFrame) -> pd.DataFrame:
    """Low: Severity 1-2 -> 0, High: Severity 3-4 -> 1 (classes 1 & 4 are extremely rare)."""
    df = df.copy()
    df[TARGET_COL_BINARY] = df[TARGET_COL_ORIGINAL].apply(lambda s: 1 if s >= 3 else 0)
    return df

### `one_hot_encode_low_cardinality()`

One-hot encodes categorical columns, keeping only the most frequent

In [10]:
def one_hot_encode_low_cardinality(df: pd.DataFrame, cols: list, top_n: int = 20) -> pd.DataFrame:
    """One-hot encodes categorical columns, keeping only the most frequent
    top_n categories per column to avoid dimensionality explosion, prefixed
    with 'enc_' to match the processed dataset naming convention."""
    df = df.copy()
    for col in cols:
        if col not in df.columns:
            continue
        top_vals = df[col].value_counts().nlargest(top_n).index
        for val in top_vals:
            safe_val = str(val).replace(" ", "_")
            df[f"enc_{col}_{val}"] = (df[col] == val).astype(int)
    return df

### `frequency_encode_high_cardinality()`

Frequency-encodes very high-cardinality identifier columns

In [11]:
def frequency_encode_high_cardinality(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    """Frequency-encodes very high-cardinality identifier columns
    (Street, City, County, Zipcode, Airport_Code, Country)."""
    df = df.copy()
    for col in cols:
        if col not in df.columns:
            continue
        freq_map = df[col].value_counts(normalize=True)
        df[f"enc_{col}_freq"] = df[col].map(freq_map).fillna(0.0)
    return df

### `encode_binary_flags()`

In [12]:
def encode_binary_flags(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    df = df.copy()
    for col in cols:
        if col not in df.columns:
            continue
        if df[col].dtype == bool:
            df[f"enc_{col}"] = df[col].astype(int)
        else:
            df[f"enc_{col}"] = df[col].astype(str).str.lower().map(
                {"true": 1, "false": 0, "1": 1, "0": 0}
            ).fillna(0).astype(int)
    return df

### `encode_twilight_and_source()`

In [13]:
def encode_twilight_and_source(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in TWILIGHT_COLS:
        if col in df.columns:
            df[f"enc_{col}_Night"] = (df[col] == "Night").astype(int)
    if "Source" in df.columns:
        top_sources = df["Source"].value_counts().nlargest(3).index
        for src in top_sources:
            df[f"enc_Source_{src}"] = (df["Source"] == src).astype(int)
    if "Timezone" in df.columns:
        top_tz = df["Timezone"].value_counts().nlargest(3).index
        for tz in top_tz:
            safe_tz = str(tz).replace("/", "_")
            df[f"enc_Timezone_{tz}"] = (df["Timezone"] == tz).astype(int)
    if "State" in df.columns:
        top_states = df["State"].value_counts().nlargest(5).index
        for st in top_states:
            df[f"enc_State_{st}"] = (df["State"] == st).astype(int)
    return df

### `apply_log_transforms()`

In [14]:
def apply_log_transforms(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    df = df.copy()
    for col in cols:
        if col in df.columns:
            df[f"{col}_log"] = np.log1p(df[col].clip(lower=0))
    return df

### `main()`

In [23]:
def main():
    in_path = "processed_full_dataset.csv"
    df = pd.read_csv(in_path)
    print(f"[LOAD] Cleaned dataset shape: {df.shape}")

    n_original = df.shape[1]

    df = add_datetime_features(df)
    df = add_road_type_features(df)
    df = add_visibility_features(df)
    df = add_speed_violation_risk_score(df)
    df = add_weather_severity_index(df)
    df = add_geo_and_text_features(df)
    df = add_target_binary(df)

    df = encode_twilight_and_source(df)
    df = one_hot_encode_low_cardinality(df, [WIND_DIR_COL, WEATHER_COL])
    df = frequency_encode_high_cardinality(df, HIGH_FREQ_CARD_COLS)
    df = encode_binary_flags(df, ROAD_INFRA_COLS)
    df = apply_log_transforms(df, LOG_TRANSFORM_COLS)

    print(f"[FEATURES] Original columns: {n_original} | Total after engineering: {df.shape[1]}")

    out_path = "processed_full_dataset.csv"
    df.to_csv(out_path, index=False)
    print(f"[SAVE] Final processed dataset -> {out_path}  shape={df.shape}")

In [24]:
import os

print(os.getcwd())
print(os.listdir())

/content
['.config', 'processed_full_dataset.csv', 'US A 10000.csv', 'sample_data']


## 3. Run

In [25]:
main()

[LOAD] Cleaned dataset shape: (10000, 144)
[FEATURES] Original columns: 144 | Total after engineering: 149
[SAVE] Final processed dataset -> processed_full_dataset.csv  shape=(10000, 149)


## 4. Inspect the result

In [27]:
proc = pd.read_csv("processed_full_dataset.csv")
print("Processed shape :", proc.shape)
print("\nBinary target balance (Severity_Binary):")
print(proc["Severity_Binary"].value_counts().sort_index().rename({0: "Low (1-2)", 1: "High (3-4)"}))

Processed shape : (10000, 149)

Binary target balance (Severity_Binary):
Severity_Binary
Low (1-2)     6053
High (3-4)    3947
Name: count, dtype: int64


Components of the composite score and the score itself:

In [28]:
score_cols = ["Is_HighSpeedRoad", "No_Traffic_Control", "Is_Night", "Low_Visibility",
              "Speed_Violation_Risk_Score", "High_Speed_Violation_Risk"]
proc[score_cols].describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
Is_HighSpeedRoad,10000.0,0.349,0.477,0.0,0.00,0.00,1.0,1.00
No_Traffic_Control,10000.0,0.674,0.469,0.0,0.00,1.00,1.0,1.00
Is_Night,10000.0,0.291,0.454,0.0,0.00,0.00,1.0,1.00
Low_Visibility,10000.0,0.000,0.000,0.0,0.00,0.00,0.0,0.00
Speed_Violation_Risk_Score,10000.0,0.363,0.242,0.0,0.25,0.25,0.6,0.85
High_Speed_Violation_Risk,10000.0,0.279,0.448,0.0,0.00,0.00,1.0,1.00
